# 02 - Tokens, context and usage

Goal: measure tokens rather than equating them with words, and budget context. Allow 25 minutes.

Run cells from top to bottom. Complete notebook 00 first. Each notebook starts with fresh state.


In [ ]:
from google.genai import types
from course_config import PROJECT_ID, MODEL_ID, EMBEDDING_LOCATION, make_client
client = make_client()
print('Project:', PROJECT_ID, '| Model:', MODEL_ID)


Tokens are model-specific pieces of text or other input. Characters, words and tokens are different units. Context includes instructions, history, tool messages and retrieved passages. Model limits and billing vary; these examples do not assume current prices.


In [ ]:
texts = ['Hello world!', 'tokenization embeddings retrieval',
         'SELECT customer_id, COUNT(*) FROM orders GROUP BY customer_id;',
         '??????? ?????']
for text in texts:
    count = client.models.count_tokens(model=MODEL_ID, contents=text)
    print({'text': text, 'characters': len(text), 'words': len(text.split()), 'tokens': count.total_tokens})


In [ ]:
prompt = 'Explain cosine similarity in 100 words.'
preflight = client.models.count_tokens(model=MODEL_ID, contents=prompt)
result = client.models.generate_content(model=MODEL_ID, contents=prompt,
    config=types.GenerateContentConfig(max_output_tokens=1200))
print('Prompt preflight:', preflight.total_tokens)
print(result.text)
print('Actual usage:', result.usage_metadata)


Usage may include thinking or cached tokens depending on the model. A short requested answer can still require a larger generation budget. max_output_tokens is a cap, not a requested answer length.


In [ ]:
# A deliberately small teaching budget, not the model's actual context limit.
input_budget = 1500
passage = 'Embeddings map text into numerical vectors. '
accepted = ''
for n in range(1, 6):
    candidate = passage * (n * 40)
    count = client.models.count_tokens(model=MODEL_ID, contents=candidate).total_tokens
    print('Repetitions:', n * 40, 'tokens:', count)
    if count > input_budget:
        break
    accepted = candidate
print('Accepted characters:', len(accepted))


## Try it yourself

Count a paragraph in English and your preferred language. Why is a universal words-to-tokens ratio unreliable? How would you allocate context among chat history and RAG evidence?
